# Module 4: Autonomous Financial Agent
In this final module, we build an agent that uses *tools* to gather data dynamically before making a decision. LangGraph is excellent for this.


In [ ]:
!pip install -q langgraph yfinance


In [ ]:
import os
import yfinance as yf
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)


## 1. Define Tools
Tools are Python functions that the LLM can decide to execute. We'll give our agent a tool to check stock prices.


In [ ]:
@tool
def get_current_price(ticker: str) -> str:
    """Gets the current or most recent closing price for a stock ticker."""
    try:
        stock = yf.Ticker(ticker)
        data = stock.history(period='1d')
        if data.empty:
            return f"Could not find price for {ticker}"
        price = data['Close'].iloc[-1]
        return f"The current price of {ticker} is {price:.2f}"
    except Exception as e:
        return str(e)

tools = [get_current_price]


## 2. Create the Agent
We use the ReAct (Reasoning and Acting) framework provided by LangGraph.


In [ ]:
agent_executor = create_react_agent(llm, tools)


## 3. Run the Agent
Let's ask the agent a question that requires it to use its tool.


In [ ]:
query = "What is the current price of Reliance Industries (RELIANCE.NS) and should I consider buying it if I want exposure to Indian conglomerates?"

# The agent will loop: reason -> act (use tool) -> observe -> reason -> final answer
for step in agent_executor.stream(
    {"messages": [("user", query)]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
